# Stage 1 — Balanced vs Unbalanced architecture backtest

This notebook reruns the architecture-selection forecast experiment from the frozen CSV snapshots.

**Primary question:** Does longer staggered-history macro coverage improve next-calendar-month excess-return forecasting relative to a shorter full-8 consistent-definition sample?

This Stage-1 notebook intentionally excludes portfolio allocation. Architecture selection is based on forecast loss on identical OOS dates.

In [ ]:
from pathlib import Path
import csv, json, math
import numpy as np

BASE=Path("/mnt/data")
CONFIG=json.loads((BASE/"stage1_backtest_config.json").read_text())

FEATURES=CONFIG["features_primary"]
ASSETS=CONFIG["assets"]

In [ ]:
def read_csv(path):
    with open(path,newline="",encoding="utf-8") as f:
        return list(csv.DictReader(f))

macro_rows=read_csv(BASE/"stage1_macro_panel_snapshot.csv")
market_rows=read_csv(BASE/"stage1_market_returns_snapshot.csv")

X_by_month={}
for r in macro_rows:
    X_by_month[r["decision_month"]]=np.array([
        float(r[f"{x}_z"]) if r[f"{x}_z"]!="" else np.nan
        for x in FEATURES
    ])

ret={a:{} for a in ASSETS}
rf={}
for r in market_rows:
    ym=r["month"]
    for a in ASSETS:
        v=r[f"{a}_total_return"]
        if v!="": ret[a][ym]=float(v)
    if r["RF_1m_tbill"]!="": rf[ym]=float(r["RF_1m_tbill"])

def next_month(ym):
    y,m=map(int,ym.split("-"))
    return f"{y+1:04d}-01" if m==12 else f"{y:04d}-{m+1:02d}"

def month_range(start,end):
    y,m=map(int,start.split("-")); ey,em=map(int,end.split("-"))
    out=[]
    while (y,m)<=(ey,em):
        out.append(f"{y:04d}-{m:02d}")
        y,m=(y+1,1) if m==12 else (y,m+1)
    return out

Y={a:{} for a in ASSETS}
for a in ASSETS:
    for dm in X_by_month:
        tm=next_month(dm)
        if tm in ret[a] and tm in rf:
            Y[a][dm]=ret[a][tm]-rf[tm]

In [ ]:
ALPHAS=np.concatenate([np.logspace(-4,10,29),[np.inf]])
INNER_MIN=24
INNER_STEP=3

def scaler(X):
    p=X.shape[1]; mu=np.zeros(p); sd=np.ones(p)
    for j in range(p):
        v=X[:,j]; mask=np.isfinite(v)
        if mask.any():
            mu[j]=v[mask].mean()
            s=v[mask].std(ddof=0)
            sd[j]=s if s>1e-12 else 1.0
    return mu,sd

def transform(X,mu,sd):
    Z=(X-mu)/sd
    Z[~np.isfinite(X)]=0.0
    return Z

def ridge_predict(Xtr,ytr,x,alpha):
    mu,sd=scaler(Xtr)
    X=transform(Xtr,mu,sd)
    xt=transform(np.atleast_2d(x),mu,sd)[0]
    ybar=float(np.mean(ytr))
    if np.isinf(alpha):
        return ybar
    beta=np.linalg.solve(X.T@X+float(alpha)*np.eye(X.shape[1]), X.T@(ytr-ybar))
    return float(ybar+xt@beta)

def choose_alpha(X,y):
    val_idx=list(range(INNER_MIN,len(y),INNER_STEP))
    losses=[]
    for a in ALPHAS:
        e=[]
        for j in val_idx:
            p=ridge_predict(X[:j],y[:j],X[j],a)
            e.append((y[j]-p)**2)
        losses.append(np.mean(e))
    best=np.min(losses)
    ix=np.where(np.array(losses)<=best+1e-12*max(1.0,abs(best)))[0]
    return ALPHAS[ix[-1]]

In [ ]:
START={"unbalanced":"2002-08","balanced":"2015-02"}
EVAL=month_range("2019-02","2022-11")

pred=[]
for arch in ["unbalanced","balanced"]:
    for asset in ASSETS:
        for dm in EVAL:
            train=[m for m in month_range(START[arch],dm) if m<dm and m in Y[asset]]
            X=np.vstack([X_by_month[m] for m in train])
            y=np.array([Y[asset][m] for m in train])
            a=choose_alpha(X,y)
            p=ridge_predict(X,y,X_by_month[dm],a)

            common=[m for m in month_range("2002-08",dm) if m<dm and m in Y[asset]]
            b=float(np.mean([Y[asset][m] for m in common]))
            pred.append((arch,asset,dm,p,Y[asset][dm],b,a))

print("Forecast rows:",len(pred))

In [ ]:
for arch in ["unbalanced","balanced"]:
    rel=[]
    for asset in ASSETS:
        rr=[r for r in pred if r[0]==arch and r[1]==asset]
        e=np.array([r[4]-r[3] for r in rr])
        be=np.array([r[4]-r[5] for r in rr])
        ratio=np.mean(e**2)/np.mean(be**2)
        rel.append(ratio)
        print(arch,asset,"Relative MSPE",round(ratio,4),"OOS R2",round(1-ratio,4))
    print(arch,"Average Relative MSPE",round(float(np.mean(rel)),4))

## Interpretation rule

If the longer Unbalanced architecture does not show a robust common-date forecast-loss improvement, retain Balanced for the main experiment because it is simpler and maintains consistent feature definitions.

The source monthly ETF return tables are rounded to one decimal percentage point, and Stage 1 uses next calendar-month returns rather than exact T+1 execution. The later global A/B/C/D backtest should use precise daily adjusted prices and exact execution timing.